# Notebook 2 — Fisher-Masked LoRA Unlearning (the fix)

**MULVA 2026 · consumes everything Notebook 1 saved to Drive.**

**What NB1's analysis told us:**
- The real collateral damage is in **`world_facts`** (general knowledge), worst in the *small* models
  under NPO (SmolLM2 retention 68% → 77% → 91% as size grows).
- `real_authors` is a floor-effect confound below 0.5B → reported separately, not in the headline metric.
- Forgetting is *confabulation* (fluent wrong answers), not degeneration — so ROUGE drops are real.

**The method — Fisher-Masked NPO (FM-NPO):**
Freeze the **retain-critical** LoRA weights during unlearning so the forgetting gradient cannot damage
general knowledge. Using NB1's saved per-weight Fisher importances, we build a boolean *keep* mask:

> protect a weight iff it is in the **top-α% most retain-important** AND **not** in the top-α% most
> forget-important → zero its gradient. Everything else trains normally under the NPO objective.

This directly targets the failure mode: retain-critical capacity is shielded, forget-specific capacity
stays free to move.

**Consumes (from `DRIVE_ROOT`):** `config.json`, `arch/<m>.json`, `ckpt/base/<m>/`,
`fisher/<m>/F_forget.pt` + `F_retain.pt`, and `metrics/*.json` (baselines to beat).
**Produces:** `ckpt/fmu/<m>/`, `metrics2/<run>.json`, `results2/rescued_pareto.{csv,png}`,
`results2/alpha_ablation.{csv,png}`.


## 1 · Mount Drive, load NB1's config, rebuild paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, pathlib
DRIVE_ROOT='/content/drive/MyDrive/mulva2026_scaling_v2'   # must match the NB1 run that holds your artifacts
assert os.path.exists(f'{DRIVE_ROOT}/config.json'), 'Run NB1 first — config.json missing.'
CONFIG=json.load(open(f'{DRIVE_ROOT}/config.json'))
for d in ['ckpt/fmu','metrics2','samples2','curves2','results2','logs']:
    pathlib.Path(DRIVE_ROOT,d).mkdir(parents=True,exist_ok=True)
MODELS=[(k,v) for fam in CONFIG['families'].values() for k,v in fam.items()]

# --- efficiency knobs (edit here, see the budget printed below) ---
SMOKE=False          # True -> run only the first model, for a ~10-min end-to-end check
FM_EPOCHS=8          # enough epochs for masked forgetting to REACH the baseline forget level
RUN_TAG='fmu8'       # distinct from the ep3 run so it neither skips nor overwrites it
ABL_TAG='abl8'
ALPHA=CONFIG['fisher']['topk_percent']   # mask sparsity for the main run (10%)
ABLATE_ALPHAS=[0,30,60]                    # 3 points, not 5 -> saves ~2 runs
ABLATE_MODEL='smollm2-360m'                # ablate where damage is large but runs are cheap
if SMOKE: MODELS=MODELS[:1]

# --- cost estimate up front ---
n_fm=len(MODELS); n_abl=len(ABLATE_ALPHAS)
print('models:',[k for k,_ in MODELS])
print(f'PLANNED RUNS: {n_fm} FM-NPO + {n_abl} ablation = {n_fm+n_abl} runs')
print(f'  ~3-5 min/run (no base training, {FM_EPOCHS} epochs, single final eval)')
print(f'  => rough budget {(n_fm+n_abl)*4} min (~{(n_fm+n_abl)*4/60:.1f} A100 hrs)')
print('  SMOKE =',SMOKE,'| set True for a one-model dry run first')

## 2 · Dependencies (same lessons as NB1: no %%capture, drop torchao, add rouge_score)

In [ ]:
!pip -q install "rouge_score==0.1.2"
!pip -q uninstall -y torchao 2>/dev/null
import importlib, rouge_score   # fail loudly here, not mid-run
print('deps ok')

## 3 · Imports, seed, logging, data (reuses NB1's detected columns)

In [ ]:
import json, time, random, datetime, functools, traceback, numpy as np, torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel
from rouge_score import rouge_scorer

def set_seed(s=CONFIG['seed']):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

LOG=f'{DRIVE_ROOT}/logs/nb2.log'
def log(*a):
    m='['+datetime.datetime.now().strftime('%H:%M:%S')+'] '+' '.join(str(x) for x in a)
    print(m); open(LOG,'a').write(m+'\n')
def manifest_add(r):
    r['ts']=datetime.datetime.now().isoformat(); open(f'{DRIVE_ROOT}/manifest.jsonl','a').write(json.dumps(r)+'\n')
def load_manifest():
    p=f'{DRIVE_ROOT}/manifest.jsonl'; return [json.loads(l) for l in open(p)] if os.path.exists(p) else []
def done(rid): return any(r.get('run_id')==rid and r.get('status')=='ok' for r in load_manifest())

# reuse NB1's detected column names
peek=json.load(open(f'{DRIVE_ROOT}/samples/data_peek.json'))
COLS=peek['columns']; log('columns from NB1:',COLS)
def load_split(n):
    ds=load_dataset(CONFIG['dataset'],n); k='train' if 'train' in ds else list(ds.keys())[0]; return ds[k]
splits={'forget':load_split(CONFIG['forget_split']),'retain':load_split(CONFIG['retain_split'])}
for u in CONFIG['utility_splits']:
    try: splits[u]=load_split(u)
    except Exception as e: log('util split missing',u,repr(e))
def format_qa(ex): return {'prompt':str(ex[COLS['q']]).strip(),'answer':str(ex[COLS['a']]).strip()}
log('splits',{k:len(v) for k,v in splits.items()})

## 4 · Evaluation harness + **corrected utility metric**
`MU_gen = harmonic_mean(retain_rougeL, world_facts)` is the headline (general capability preserved).
`real_authors` is recorded but kept out of the headline below 0.5B (NB1 floor-effect finding).

In [ ]:
_scorer=rouge_scorer.RougeScorer(['rougeL'],use_stemmer=True)
def rougeL(p,g): return _scorer.score(g,p)['rougeL'].recall
def hmean(xs):
    xs=[x for x in xs if x is not None]
    return 0.0 if any(x<=0 for x in xs) else len(xs)/sum(1/x for x in xs)

@torch.no_grad()
def generate_batch(model,tok,prompts,max_new=96,bs=16):
    outs=[]
    for i in range(0,len(prompts),bs):
        chunk=prompts[i:i+bs]
        msgs=[tok.apply_chat_template([{'role':'user','content':p}],tokenize=False,add_generation_prompt=True) for p in chunk]
        enc=tok(msgs,return_tensors='pt',padding=True,truncation=True,max_length=512).to(model.device)
        gen=model.generate(**enc,max_new_tokens=max_new,do_sample=False,pad_token_id=tok.pad_token_id or tok.eos_token_id)
        for j,g in enumerate(gen):
            outs.append(tok.decode(g[enc['input_ids'].shape[1]:],skip_special_tokens=True).strip())
    return outs

def _rl(split,n,model,tok):
    rows=[format_qa(split[i]) for i in range(min(n,len(split)))]
    preds=generate_batch(model,tok,[r['prompt'] for r in rows])
    return float(np.mean([rougeL(p,r['answer']) for p,r in zip(preds,rows)])), list(zip(
        [r['prompt'] for r in rows[:8]],preds[:8],[r['answer'] for r in rows[:8]]))

def eval_model(model,tok,run_id,n_forget=200,n_retain=200,n_util=100,save=True):
    fe,fs=_rl(splits['forget'],n_forget,model,tok)
    ru,rs=_rl(splits['retain'],n_retain,model,tok)
    util={u:_rl(splits[u],n_util,model,tok)[0] for u in CONFIG['utility_splits'] if u in splits}
    wf=util.get('world_facts'); mu_gen=hmean([ru,wf]) if wf is not None else ru
    m={'run_id':run_id,'forget_rougeL':fe,'retain_rougeL':ru,'utility':util,
       'MU_gen':mu_gen,'MU':float(np.mean([ru]+list(util.values())))}  # MU kept for back-compat
    if save:
        json.dump(m,open(f'{DRIVE_ROOT}/metrics2/{run_id}.json','w'),indent=2)
        json.dump({'forget':fs,'retain':rs},open(f'{DRIVE_ROOT}/samples2/{run_id}.json','w'),indent=2)
    log('EVAL',run_id,f'forget={fe:.3f} retain={ru:.3f} world={util.get("world_facts",0):.3f} MU_gen={mu_gen:.3f}')
    return m

@torch.no_grad()
def quick_forget(model,tok,n=40):
    # cheap matched-forget probe: few samples, short generations
    rows=[format_qa(splits['forget'][i]) for i in range(min(n,len(splits['forget'])))]
    preds=generate_batch(model,tok,[r['prompt'] for r in rows],max_new=48)
    return float(np.mean([rougeL(p,r['answer']) for p,r in zip(preds,rows)]))

## 5 · Loaders + SFT batching (identical formatting to NB1)

In [ ]:
from torch.utils.data import DataLoader
def load_base(hf_id):
    tok=AutoTokenizer.from_pretrained(hf_id,trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token=tok.eos_token
    tok.padding_side='left'
    model=AutoModelForCausalLM.from_pretrained(hf_id,torch_dtype=torch.bfloat16,trust_remote_code=True,device_map='cuda')
    return model,tok
def make_loader(tok,rows,bs):
    def collate(batch):
        texts=[tok.apply_chat_template([{'role':'user','content':r['prompt']},
               {'role':'assistant','content':r['answer']}],tokenize=False) for r in batch]
        enc=tok(texts,return_tensors='pt',padding=True,truncation=True,max_length=CONFIG['memorize']['max_len'])
        enc['labels']=enc['input_ids'].clone(); return enc
    data=[format_qa(r) for r in rows]
    return DataLoader(data,batch_size=bs,shuffle=True,collate_fn=collate)
def seq_logprob(model,batch):
    out=model(input_ids=batch['input_ids'],attention_mask=batch['attention_mask'])
    logp=F.log_softmax(out.logits[:,:-1],dim=-1); tgt=batch['input_ids'][:,1:].unsqueeze(-1)
    lp=logp.gather(-1,tgt).squeeze(-1); mask=batch['attention_mask'][:,1:]
    return (lp*mask).sum(-1)/mask.sum(-1).clamp(min=1)

## 6 · Build the retain-protect gradient mask from NB1's Fisher importances
We load the raw per-weight importances (`F_forget.pt`, `F_retain.pt`) — saved by NB1 precisely so the
mask can be rebuilt at **any** α without recomputation. `keep_mask[name]` is 1.0 everywhere except the
protected weights (retain-critical ∧ not forget-critical), where it is 0.0.

In [ ]:
def build_keep_masks(model_key, alpha_pct):
    """Return {param_name: float keep-mask} protecting retain-critical, non-forget weights."""
    Ff=torch.load(f'{DRIVE_ROOT}/fisher/{model_key}/F_forget.pt', map_location='cpu')
    Fr=torch.load(f'{DRIVE_ROOT}/fisher/{model_key}/F_retain.pt', map_location='cpu')
    keep={}; n_prot=0; n_tot=0
    for name in Fr:
        fr=Fr[name].float().flatten(); ff=Ff[name].float().flatten()
        k=max(1,int(len(fr)*alpha_pct/100))
        retain_crit=torch.zeros_like(fr,dtype=torch.bool); retain_crit[torch.topk(fr,k).indices]=True
        forget_crit=torch.zeros_like(ff,dtype=torch.bool); forget_crit[torch.topk(ff,k).indices]=True
        protect=retain_crit & (~forget_crit)
        keep[name]=(~protect).float().view_as(Fr[name])
        n_prot+=int(protect.sum()); n_tot+=protect.numel()
    log(f'  mask alpha={alpha_pct}% -> protecting {n_prot}/{n_tot} ({100*n_prot/n_tot:.1f}%) weights')
    return keep

def attach_mask_hooks(model, keep):
    """Register grad hooks: grad <- grad * keep_mask. Returns handles to remove later."""
    handles=[]
    name2p=dict(model.named_parameters())
    for name,m in keep.items():
        if name in name2p and name2p[name].requires_grad:
            p=name2p[name]; msk=m.to(p.device,dtype=p.dtype)
            handles.append(p.register_hook(lambda g,msk=msk: g*msk))
    return handles

## 7 · FM-NPO trainer (masked unlearning, matched-forget early stop)
Loads NB1's memorized base adapter, attaches the keep-mask hooks, runs the NPO objective. Optional
early-stop when forget ROUGE-L reaches `target_forget` so FM-NPO is compared to the baseline **at
matched forgetting**, isolating the utility difference.

In [ ]:
def fm_npo(model_key, hf_id, alpha_pct, target_forget=None, tag='fmu', epochs=None):
    run_id=f'{tag}_a{alpha_pct}__{model_key}'
    if done(run_id): log('SKIP',run_id); return
    t0=time.time(); log('START',run_id)
    try:
        set_seed()
        base,tok=load_base(hf_id); tok.padding_side='right'
        model=PeftModel.from_pretrained(base,f'{DRIVE_ROOT}/ckpt/base/{model_key}',is_trainable=True)
        keep=build_keep_masks(model_key, alpha_pct)
        handles=attach_mask_hooks(model, keep)
        ucfg=CONFIG['unlearn']; beta=ucfg['npo_beta']
        E=epochs or FM_EPOCHS   # capped: no base training here, only shift forget-specific weights
        f_loader=make_loader(tok,list(splits['forget']),ucfg['batch_size'])
        r_loader=make_loader(tok,list(splits['retain']),ucfg['batch_size'])
        opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=ucfg['lr'])
        curve=[]; step=0; r_it=iter(r_loader); stopped=False
        for ep in range(E):
            for fb in f_loader:
                fb={k:v.cuda() for k,v in fb.items()}
                try: rb=next(r_it)
                except StopIteration: r_it=iter(r_loader); rb=next(r_it)
                rb={k:v.cuda() for k,v in rb.items()}
                retain_loss=model(**rb).loss
                with model.disable_adapter(), torch.no_grad(): ref=seq_logprob(model,fb)
                pol=seq_logprob(model,fb)
                npo=-(2/beta)*F.logsigmoid(-beta*(pol-ref)).mean()
                loss=npo+ucfg['retain_weight']*retain_loss
                (loss/ucfg['grad_accum']).backward(); step+=1
                if step%ucfg['grad_accum']==0: opt.step(); opt.zero_grad()
                if step%20==0: curve.append({'step':step,'loss':float(loss)})
            # matched-forget early stop
            if target_forget is not None:
                tok.padding_side='left'; model.eval()
                fnow=quick_forget(model,tok); model.train(); tok.padding_side='right'
                log(f'  {run_id} ep{ep} forget~{fnow:.3f}')
                if fnow<=target_forget: log('  reached target forget'); stopped=True
            if stopped: break
        for h in handles: h.remove()
        import pandas as pd; pd.DataFrame(curve).to_csv(f'{DRIVE_ROOT}/curves2/{run_id}.csv',index=False)
        model.save_pretrained(f'{DRIVE_ROOT}/ckpt/fmu/{model_key}' if tag=='fmu' else f'{DRIVE_ROOT}/ckpt/fmu/{run_id}')
        tok.padding_side='left'; model.eval(); eval_model(model,tok,run_id)
        manifest_add({'run_id':run_id,'status':'ok','secs':round(time.time()-t0,1),'alpha':alpha_pct})
        log('OK',run_id,f'({round(time.time()-t0)}s)')
    except Exception as e:
        open(f'{DRIVE_ROOT}/logs/errors.jsonl','a').write(json.dumps(
            {'run_id':run_id,'error':repr(e),'trace':traceback.format_exc()})+'\n')
        manifest_add({'run_id':run_id,'status':'error','error':repr(e)}); log('ERROR',run_id,repr(e)); raise
    finally:
        import gc; gc.collect(); torch.cuda.empty_cache()

## 8 · Phase D — Run FM-NPO across all scales
Default α = NB1's `fisher.topk_percent` (10%). To compare at matched forgetting, set `target_forget`
per model from the baseline NPO forget ROUGE-L (read from NB1 metrics).

In [ ]:
def baseline_forget(model_key, method='npo'):
    p=f'{DRIVE_ROOT}/metrics/{method}__{model_key}.json'
    return json.load(open(p))['forget_rougeL'] if os.path.exists(p) else None

for k,hf in MODELS:
    tgt=baseline_forget(k,'npo')            # match NPO's forgetting level
    fm_npo(k,hf,ALPHA,target_forget=tgt,tag=RUN_TAG)

## 9 · Phase E — α ablation on one representative small model
Smaller models are where the damage lives, so ablate there (default SmolLM2-360M). Sweeps the mask
sparsity α; each run early-stops at the matched forget level so only utility varies.

In [ ]:
ALL_IDS={k:v for fam in CONFIG['families'].values() for k,v in fam.items()}
hf=ALL_IDS[ABLATE_MODEL]
tgt=baseline_forget(ABLATE_MODEL,'npo')
for a in ABLATE_ALPHAS:            # 3 points (0/30/60); the main run already covers alpha=10
    fm_npo(ABLATE_MODEL,hf,a,target_forget=tgt,tag=ABL_TAG)

## 10 · Phase F — Rescued-Pareto comparison (baseline vs FM-NPO)
Pulls NB1 baseline metrics + NB2 FM metrics, aligns them, and reports the headline:
**`world_facts` retention at matched forgetting**. Saves the figure that is the paper's money plot.

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt

def collect(dirn):
    rows=[]
    for f in glob.glob(f'{DRIVE_ROOT}/{dirn}/*.json'):
        m=json.load(open(f)); rid=m['run_id']
        rows.append({'run_id':rid,'forget':m['forget_rougeL'],'retain':m['retain_rougeL'],
                     'world':m['utility'].get('world_facts'),'MU_gen':m.get('MU_gen')})
    return rows
base_rows=collect('metrics'); fm_rows=collect('metrics2')
df=pd.DataFrame(base_rows+fm_rows)
df.to_csv(f'{DRIVE_ROOT}/results2/all_metrics.csv',index=False)

# headline: world_facts retention vs scale, baseline NPO vs FM-NPO (per family)
import re
def key(rid): return rid.split('__')[-1]
rows=[]
for k,_ in MODELS:
    b=f'{DRIVE_ROOT}/metrics/base__{k}.json'; n=f'{DRIVE_ROOT}/metrics/npo__{k}.json'
    fms=sorted(glob.glob(f'{DRIVE_ROOT}/metrics2/{RUN_TAG}_a*__{k}.json'))
    if not(os.path.exists(b) and os.path.exists(n) and fms): continue
    wb=json.load(open(b))['utility']['world_facts']
    wn=json.load(open(n))['utility']['world_facts']
    wf=json.load(open(fms[0]))['utility']['world_facts']
    rows.append({'model':k,'family':k.split('-')[0],'base_world':wb,
                 'npo_ret%':100*wn/wb,'fmnpo_ret%':100*wf/wb})
comp=pd.DataFrame(rows); comp.to_csv(f'{DRIVE_ROOT}/results2/rescued_pareto.csv',index=False)
print(comp.to_string(index=False))

fig,ax=plt.subplots(figsize=(7,5))
for fam,g in comp.groupby('family'):
    g=g.sort_values('base_world')
    ax.plot(range(len(g)),g['npo_ret%'],'o--',label=f'{fam} NPO')
    ax.plot(range(len(g)),g['fmnpo_ret%'],'s-',label=f'{fam} FM-NPO')
ax.set_ylabel('world_facts retention (% of base)'); ax.set_xlabel('model scale (small -> large)')
ax.set_title('FM-NPO rescues general knowledge in small models'); ax.legend(); fig.tight_layout()
fig.savefig(f'{DRIVE_ROOT}/results2/rescued_pareto.png',dpi=150); plt.show()

## 11 · α-ablation plot + final readiness summary

In [ ]:
import glob, json, pandas as pd, matplotlib.pyplot as plt
abl=[]
for f in sorted(glob.glob(f'{DRIVE_ROOT}/metrics2/{ABL_TAG}_a*__{ABLATE_MODEL}.json')):
    m=json.load(open(f)); a=int(f.split('_a')[1].split('__')[0])
    abl.append({'alpha':a,'forget':m['forget_rougeL'],'world':m['utility'].get('world_facts'),'MU_gen':m.get('MU_gen')})
abl=pd.DataFrame(abl).sort_values('alpha'); abl.to_csv(f'{DRIVE_ROOT}/results2/alpha_ablation.csv',index=False)
print(abl.to_string(index=False))
if len(abl):
    fig,ax=plt.subplots(figsize=(7,4))
    ax.plot(abl['alpha'],abl['world'],'s-',label='world_facts (keep high)')
    ax.plot(abl['alpha'],abl['forget'],'o--',label='forget (keep low)')
    ax.set_xlabel('mask sparsity alpha (%)'); ax.set_ylabel('ROUGE-L'); ax.set_title(f'alpha ablation ({ABLATE_MODEL})')
    ax.legend(); fig.tight_layout(); fig.savefig(f'{DRIVE_ROOT}/results2/alpha_ablation.png',dpi=150); plt.show()
print('\n=== NB2 DONE ===  artifacts in results2/, metrics2/, ckpt/fmu/')